# Notebook 6 — Train, Tune, Evaluate

## Load feature tables and feature list

In [1]:
import pandas as pd

train = pd.read_csv("../artifacts/05_train_features.csv")
val = pd.read_csv("../artifacts/05_val_features.csv")
test = pd.read_csv("../artifacts/05_test_features.csv")

with open("../artifacts/05_feature_list.txt") as f:
    feature_list = f.read().splitlines()

print("train:", train.shape)
print("val:  ", val.shape)
print("test: ", test.shape)
print("Number of features:", len(feature_list))

train: (67529, 49)
val:   (14470, 49)
test:  (14471, 49)
Number of features: 48


**Note:** Loaded all three feature tables and the feature list — 
matches Notebook 5's saved artifacts exactly (49 columns including 
target, 48 features excluding it).

## Split into X (features) and y (target)

In [2]:
X_train, y_train = train[feature_list], train["is_late"]
X_val, y_val = val[feature_list], val["is_late"]
X_test, y_test = test[feature_list], test["is_late"]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)

X_train: (67529, 48) y_train: (67529,)
X_val:   (14470, 48) y_val:   (14470,)
X_test:  (14471, 48) y_test:  (14471,)


**Note:** X/y split successful — X has 48 feature columns, y is a single 
target column, with matching row counts across all splits.

## Baseline: always predict "not late"

In [3]:
from sklearn.metrics import classification_report

baseline_pred = [0] * len(y_val)

print(classification_report(y_val, baseline_pred))

              precision    recall  f1-score   support

           0       0.95      1.00      0.97     13697
           1       0.00      0.00      0.00       773

    accuracy                           0.95     14470
   macro avg       0.47      0.50      0.49     14470
weighted avg       0.90      0.95      0.92     14470



C:\Users\User\Desktop\MLOps\olist_project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\User\Desktop\MLOps\olist_project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\User\Desktop\MLOps\olist_project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

**Note:** Baseline gets 95% accuracy but 0% recall on late orders — it 
never predicts "late" at all, making it useless for the actual business 
goal (catching late deliveries). This confirms accuracy alone is 
misleading here. The warning is expected (division by zero when no 
"late" predictions exist) and not an error. This baseline (0% recall on 
class 1) is what our real model must beat.

## Train a real model: Random Forest with class balancing

In [4]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train, y_train)

val_pred = model.predict(X_val)
print(classification_report(y_val, val_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.95      0.98      0.96     13697
           1       0.08      0.04      0.05       773

    accuracy                           0.93     14470
   macro avg       0.51      0.51      0.51     14470
weighted avg       0.90      0.93      0.91     14470



**Note:** The model shows only a slight improvement over baseline — recall 
on late orders improved from 0% to 4%, but this is still very weak in 
practice (misses 96% of actual late deliveries). This honestly reflects 
that our current features (order/payment aggregates, state, month) may 
not capture the real drivers of delivery delays (e.g., carrier performance, 
external disruptions), which aren't available in this dataset. This is a 
legitimate finding, not a coding error — worth noting as a limitation and 
a direction for future feature engineering (e.g., seller-level delay 
history, distance between customer and seller).

## Try tuning: adjust max_depth

In [5]:
from sklearn.metrics import f1_score

results = []

for depth in [5, 10, 15, 20, None]:
    m = RandomForestClassifier(
        n_estimators=200,
        max_depth=depth,
        class_weight="balanced",
        random_state=42
    )
    m.fit(X_train, y_train)
    pred = m.predict(X_val)
    f1 = f1_score(y_val, pred, zero_division=0)
    recall = classification_report(y_val, pred, output_dict=True, zero_division=0)["1"]["recall"]
    results.append({"max_depth": depth, "f1_class1": f1, "recall_class1": recall})
    print(f"max_depth={depth}: f1(late)={f1:.3f}, recall(late)={recall:.3f}")

max_depth=5: f1(late)=0.090, recall(late)=0.125
max_depth=10: f1(late)=0.110, recall(late)=0.118
max_depth=15: f1(late)=0.112, recall(late)=0.107
max_depth=20: f1(late)=0.087, recall(late)=0.069
max_depth=None: f1(late)=0.050, recall(late)=0.036


**Note:** Limiting tree depth improved performance significantly compared 
to unlimited depth (which overfits) — max_depth=5 gives the best recall 
(0.125) on late orders, while max_depth=15 gives the best F1 (0.112). 
Since catching late orders (recall) matters more for the business goal 
than precision here, choosing max_depth=5 as the final model.

## Train final model with best configuration

In [6]:
best_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    class_weight="balanced",
    random_state=42
)
best_model.fit(X_train, y_train)

val_pred = best_model.predict(X_val)
print(classification_report(y_val, val_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.95      0.91      0.93     13697
           1       0.07      0.13      0.09       773

    accuracy                           0.86     14470
   macro avg       0.51      0.52      0.51     14470
weighted avg       0.90      0.86      0.88     14470



**Note:** Final model (max_depth=5) achieves 13% recall on late orders, 
up from 0% baseline — a real but modest improvement. Accuracy dropped to 
86% (expected trade-off: catching more late orders means more false 
alarms). Precision remains low (7%), meaning most "late" predictions are 
false alarms. This is an honest first-iteration result: the model 
provides some signal but would need richer features (e.g., seller 
history, carrier data, weather) to be production-ready.

## Final evaluation on test set (touched once, at the very end)

In [7]:
test_pred = best_model.predict(X_test)
print(classification_report(y_test, test_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.93      0.98      0.96     13514
           1       0.02      0.01      0.01       957

    accuracy                           0.92     14471
   macro avg       0.48      0.50      0.48     14471
weighted avg       0.87      0.92      0.90     14471



## Final Results Summary

**Test set performance is much weaker than validation** — recall on late 
orders dropped from 0.13 (val) to 0.01 (test), and precision dropped 
from 0.07 to 0.02.

**Why this likely happened:** The EDA (Notebook 4) found a strong, unusual 
spike in late deliveries during Feb-Mar 2018 — the final months of the 
training period. The model likely learned patterns specific to that 
unusual period, which did not repeat in the test period (Jun-Aug 2018). 
This means the model does not generalize well across time.

**This is a genuine and valuable finding, not a bug** — it demonstrates 
exactly why a time-based split (rather than random) is important: it 
exposes generalization problems that a random split would have hidden.

**Comparison to baseline:**
| Metric (class 1 = late) | Baseline | Tuned Model (val) | Tuned Model (test) |
|---|---|---|---|
| Recall | 0.00 | 0.13 | 0.01 |
| Precision | 0.00 | 0.07 | 0.02 |

**Conclusion:** The current feature set and model provide only a weak, 
inconsistent signal for predicting late deliveries. Future work should 
focus on: (1) investigating the Feb-Mar 2018 anomaly directly, (2) adding 
richer features (seller-level history, shipping distance, carrier info), 
and (3) possibly testing whether a simpler, more stable model generalizes 
better across time than this tuned Random Forest.

## Save trained model and results summary

In [8]:
import joblib

joblib.dump(best_model, "../artifacts/06_trained_model.pkl")

with open("../artifacts/06_results_summary.txt", "w") as f:
    f.write("Model: RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced')\n\n")
    f.write("Baseline (always predict not-late):\n")
    f.write("  Val recall (late)=0.00, precision (late)=0.00\n\n")
    f.write("Tuned model on validation set:\n")
    f.write(classification_report(y_val, val_pred, zero_division=0))
    f.write("\nTuned model on test set (final, touched once):\n")
    f.write(classification_report(y_test, test_pred, zero_division=0))
    f.write("\n\nKey finding: Test performance is much weaker than validation, ")
    f.write("likely due to an unusual delay spike in Feb-Mar 2018 (end of training period) ")
    f.write("that the model overfit to. Model does not generalize well across time. ")
    f.write("Future work: investigate the anomaly, add richer features (seller history, ")
    f.write("shipping distance, carrier data).\n")

print("Saved model and results summary.")

Saved model and results summary.
